# 01 · Ingest raw data

**Goal:** read every raw source file exactly as downloaded and stack it into one tidy table with the columns
`series_id · obs_date · value`. Anything unusual is written to an **issue log** instead of being silently fixed.

> **Series:** 01 Ingest → 02 Align & transform → 03 Event windows → 04 Database → 05 Results report → 06 Analysis & charts. Run them in order (or run `python run_pipeline.py` from the project folder). Paths and settings live in `config.py`.

**Rules followed in this notebook**
- Raw files in `01_Data/1_Raw/` are never edited.
- A blank, `.` or `-` means *no observation*. It is never turned into zero. The one exception is RBI's intervention table, where RBI itself defines `-` as *nil*.
- If a date appears twice, keep the first and log it.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

import io
import json
import re
import zipfile
import pandas as pd
from config import (PROC, ANALYSIS_START, ANALYSIS_END, read_reference, REF_REPO, RAW_RBI_REFERENCE,
                    RAW_RBI_INTERBANK, RAW_RBI_CPI, RAW_RBI_INTERVENTION, RAW_FRED_DAILY, RAW_FRED_MONTHLY,
                    RAW_FRED_USDINR, RAW_LBMA_GOLD, RAW_NIFTY_DIR)

## Helper functions

`log()` records an issue, `to_num()` turns text into numbers (blanks become missing), and `tidy()` converts any raw table
into the standard long format. `tidy()` also counts gaps. It only counts **inside** the series' own date range and only on
**weekdays**, because the wide FRED files are padded with blanks outside each series' range.

In [2]:
issues = []


def log(series, rule, detail, severity="info", obs_date=None, action=""):
    issues.append(dict(stage="ingest", series_id=series, obs_date=obs_date,
                       rule=rule, detail=detail, severity=severity, action=action))


def to_num(s):
    return pd.to_numeric(s.astype(str).str.strip().replace({"": None, ".": None, "-": None, "nan": None}),
                         errors="coerce")


def tidy(df, series_id, date_col, value_col):
    out = pd.DataFrame({"series_id": series_id,
                        "obs_date": pd.to_datetime(df[date_col]),
                        "value": to_num(df[value_col])})
    valid = out["value"].notna()
    if valid.any():
        # count gaps only inside the series' own date range and on weekdays
        # (wide FRED bundles are padded with blanks outside each series' range)
        inside = out["obs_date"].between(out.loc[valid, "obs_date"].min(), out.loc[valid, "obs_date"].max())
        daily = out.loc[valid, "obs_date"].diff().dt.days.median() <= 3
        mask = ~valid & inside & ((out["obs_date"].dt.dayofweek < 5) if daily else True)
        n_null = int(mask.sum())
        if n_null:
            log(series_id, "null_values",
                f"{n_null} {'weekday' if daily else 'period'} rows inside the series range have no value "
                "(holidays / not published)",
                "info", action="dropped (treated as no observation, never as zero)")
    out = out.dropna(subset=["value"])
    dup = out.duplicated(["series_id", "obs_date"], keep="first")
    if dup.any():
        for d in out.loc[dup, "obs_date"]:
            log(series_id, "duplicate_date", "duplicate observation", "warning", d.date(), "kept first")
        out = out[~dup]
    return out

## 1 · USD/INR: RBI reference rate (primary series)

Source: RBI DBIE → *Daily Exchange Rate of the Indian Rupee*. This is the official RBI/FBIL reference rate (₹ per US$),
fixed around 13:30 IST. Holidays appear as `-` and become non-trading days.

In [3]:
# ---------------------------------------------------------------- RBI reference rate (PRIMARY)
def load_rbi_reference():
    """DBIE 'Daily Exchange Rate of the Indian Rupee' - RBI/FBIL reference rates (Rs per unit)."""
    path = RAW_RBI_REFERENCE
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    start = next(i for i, l in enumerate(lines) if l.startswith("Date\t"))
    recs = [l.split("\t")[:2] for l in lines[start + 1:] if re.match(r"^\d{1,2}-[A-Za-z]{3}-\d{4}\t", l)]
    df = pd.DataFrame(recs, columns=["Date", "usd"])
    df["Date"] = pd.to_datetime(df["Date"].str.strip(), format="%d-%b-%Y")
    hol = df.loc[df["usd"].str.strip().eq("-"), ["Date"]].rename(columns={"Date": "obs_date"})
    hol.to_csv(PROC / "rbi_holiday_rows.csv", index=False)
    log("USDINR_RBI", "holiday_rows", f"{len(hol)} dates listed with '-' (no reference rate)", "info",
        action="treated as non-trading days")
    return tidy(df, "USDINR_RBI", "Date", "usd")

In [4]:
rbi_ref = load_rbi_reference()
print(f"{len(rbi_ref):,} daily observations, {rbi_ref.obs_date.min():%d %b %Y} to {rbi_ref.obs_date.max():%d %b %Y}")
rbi_ref.tail()

6,792 daily observations, 25 Aug 1998 to 29 Sep 2026


,series_id,obs_date,value
6790,USDINR_RBI,1998-09-02,42.5297
6791,USDINR_RBI,1998-09-01,42.5206
6792,USDINR_RBI,1998-08-28,42.5405
6793,USDINR_RBI,1998-08-27,42.5297
6794,USDINR_RBI,1998-08-25,42.5007


## 2 · USD/INR: RBI interbank table (cross-check)

DBIE's *Daily Interbank Forex Rates* table has open/high/low/close **and its own copy of the reference rate**, which is
used to cross-check the primary series.

> **Quirk worth knowing:** RBI labels "High" and "Low" from the **rupee's** side. "High" means the *strongest* rupee,
> i.e. the lowest ₹-per-$ number. The code checks this instead of assuming it.

In [5]:
# ---------------------------------------------------------------- RBI (DBIE)
def load_rbi():
    path = RAW_RBI_INTERBANK
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    start = next(i for i, l in enumerate(lines) if l.startswith("Date\t"))
    df = pd.read_csv(io.StringIO("\n".join(lines[start:])), sep="\t", dtype=str)
    df.columns = [c.strip() for c in df.columns]
    df["Date"] = pd.to_datetime(df["Date"].str.strip(), format="%d %B %Y")

    ref = df["RBI Reference Rate"].str.strip()

    wk = (df["Date"].dt.dayofweek >= 5) & ref.ne("-")
    for d in df.loc[wk, "Date"]:
        log("USDINR_RBI_IBTABLE", "weekend_value", "value on a Saturday in the interbank table",
            "warning", d.date(), "not used - primary series comes from the Daily Exchange Rate table")

    # RBI labels High/Low from the rupee's side: 'High' is the strongest rupee
    # (= lowest INR per USD). Check and record instead of silently swapping.
    o = to_num(df["Interbank High Rate"]); l = to_num(df["Interbank Low Rate"])
    swapped = (o < l).mean()
    log("USDINR_RBI_CLOSE", "high_low_convention",
        f"'High' < 'Low' on {swapped:.0%} of rows - RBI quotes High/Low in rupee terms", "info",
        action="columns renamed inr_strongest / inr_weakest in fact_fx")

    rbi = pd.concat([
        tidy(df, "USDINR_RBI_IBTABLE", "Date", "RBI Reference Rate"),
        tidy(df, "USDINR_RBI_CLOSE", "Date", "Interbank Close Rate"),
        tidy(df, "USDINR_RBI_OPEN", "Date", "Interbank Open Rate"),
        tidy(df, "USDINR_RBI_INRSTRONG", "Date", "Interbank High Rate"),
        tidy(df, "USDINR_RBI_INRWEAK", "Date", "Interbank Low Rate"),
    ])
    return rbi

In [6]:
rbi_ib = load_rbi()
rbi_ib.groupby("series_id")["obs_date"].agg(["min", "max", "count"])

,min,max,count
series_id,,,
USDINR_RBI_CLOSE,2000-08-08,2026-06-30,6114
USDINR_RBI_IBTABLE,2000-08-08,2026-06-30,6259
USDINR_RBI_INRSTRONG,2000-08-08,2026-06-30,6115
USDINR_RBI_INRWEAK,2000-08-08,2026-06-30,6116
USDINR_RBI_OPEN,2000-08-08,2026-06-30,6108


## 3 · India CPI (MoSPI, via RBI DBIE)

The export holds **three base-year blocks** (2010, 2012 and 2024 = 100). MoSPI switched to base 2024 in January 2026.
Index levels cannot be compared across a base change, so the analysis uses the **published YoY inflation** and stores
the base year on every row.

In [7]:
# ---------------------------------------------------------------- India CPI (MoSPI via RBI DBIE)
def load_cpi_mospi():
    """All-India CPI-Combined General Index and published YoY inflation.

    The DBIE export holds three base-year blocks (2010, 2012, 2024 = 100).
    Headline inflation = the figure officially published at the time:
      2010-base block before Jan-2014, 2012-base block Jan-2014 .. Dec-2025,
      2024-base block from Jan-2026 (MoSPI's new series).
    The base year is kept on every row so the series break is visible.
    """
    path = RAW_RBI_CPI
    if not path.exists():
        log("IN_CPI_MOSPI", "missing_source", "CPI file not found", "error")
        return pd.DataFrame(columns=["series_id", "obs_date", "value"])
    L = [l.rstrip("\r") for l in path.read_text(encoding="utf-8", errors="replace").split("\n")]
    blocks = []
    for h in [i for i, l in enumerate(L) if l.startswith("Month\t")]:
        base = next((re.search(r"(\d{4})=100", L[k]).group(1) for k in range(h - 5, h)
                     if re.search(r"Base\s*:\s*\d{4}=100", L[k])), None)
        grp, sec, mea = (L[h + i].split("\t") for i in range(3))
        def ffill(v):
            out, last = [], ""
            for x in v:
                last = x.strip() or last
                out.append(last)
            return out
        grp, sec = ffill(grp), ffill(sec)
        cols = {}
        for j in range(3, len(grp)):
            if "general index" in grp[j].lower() and sec[j] == "Combined" and j < len(mea):
                cols["index" if mea[j].strip() == "Index" else "yoy"] = j
        k, rows = h + 3, []
        while k < len(L) and not L[k].startswith("Note"):
            p = L[k].split("\t")
            if len(p) > 2 and p[1].strip() == "ALL INDIA":
                g = lambda c: p[cols[c]].strip() if c in cols and cols[c] < len(p) else ""
                rows.append(dict(month=pd.to_datetime(p[0].strip(), format="%b-%Y"), base=base,
                                 status=p[2].strip(), index=g("index"), yoy=g("yoy")))
            k += 1
        b = pd.DataFrame(rows)
        b["index"] = pd.to_numeric(b["index"], errors="coerce")
        b["yoy"] = pd.to_numeric(b["yoy"], errors="coerce")
        blocks.append(b)
        log("IN_CPI_MOSPI", "cpi_block", f"base {base}=100: {len(b)} All-India rows "
            f"{b.month.min():%b-%Y}..{b.month.max():%b-%Y}, {int(b.yoy.notna().sum())} with published YoY", "info")
    allb = pd.concat(blocks)
    pick = pd.concat([allb[(allb.base == "2010") & (allb.month < "2014-01-01")],
                      allb[(allb.base == "2012") & (allb.month.between("2014-01-01", "2025-12-01"))],
                      allb[(allb.base == "2024") & (allb.month >= "2026-01-01")]]).sort_values("month")
    pick.to_csv(PROC / "cpi_mospi.csv", index=False, date_format="%Y-%m-%d")
    log("IN_CPI_MOSPI", "series_break",
        "CPI base changes 2010->2012 (Jan-2014) and 2012->2024 (Jan-2026); index LEVELS are not comparable across the "
        "break, published YoY inflation is used for charts", "warning",
        action="base year stored per row (fact_macro.series_version)")
    return tidy(pick.rename(columns={"month": "obs_date", "index": "value"}),
                "IN_CPI_MOSPI", "obs_date", "value")

In [8]:
cpi = load_cpi_mospi()
cpi_detail = pd.read_csv(PROC / "cpi_mospi.csv")
cpi_detail.tail(10)   # Aug-2026 YoY should read 4.82% - same as the RBI DBIE homepage

,month,base,status,index,yoy
178,2025-11-01,2012,Final,197.90,0.71
179,2025-12-01,2012,Provisional,198.00,1.33
180,2026-01-01,2024,Final,104.45,2.73
181,2026-02-01,2024,Final,104.57,3.21
182,2026-03-01,2024,Final,104.84,3.40
183,2026-04-01,2024,Final,105.12,3.48
184,2026-05-01,2024,Final,105.91,3.94
185,2026-06-01,2024,Final,107.00,4.38
186,2026-07-01,2024,Final,107.95,4.45
187,2026-08-01,2024,Provisional,108.74,4.82


## 4 · RBI foreign-exchange intervention

RBI Bulletin **Table 4: Sale/Purchase of US Dollar by the RBI**. Monthly, US$ million, and it includes swap and forward
legs by value date. A negative *net* figure means RBI **sold** dollars, which supports the rupee.

In [9]:
# ---------------------------------------------------------------- RBI FX intervention
def load_rbi_intervention():
    """RBI Bulletin Table 4: Sale/Purchase of US Dollar by the RBI (monthly, US$ million)."""
    path = RAW_RBI_INTERVENTION
    if not path.exists():
        log("IN_RBI_FX_NET", "missing_source", "intervention file not found", "error")
        return pd.DataFrame(columns=["series_id", "obs_date", "value"])
    import warnings
    with warnings.catch_warnings():            # RBI's xlsx has no default style - harmless
        warnings.simplefilter("ignore", UserWarning)
        raw = pd.read_excel(path, sheet_name=0, header=None)
    hdr = raw.index[raw.iloc[:, 1].astype(str).str.strip().eq("Month")][0]
    body = raw.iloc[hdr + 2:, [1, 3, 4, 5, 9]]
    body.columns = ["month", "net", "purchase", "sale", "fwd_book"]
    body = body[pd.to_datetime(body["month"], errors="coerce").notna()].copy()
    # RBI statistical convention: '-' in this table means NIL (no purchase / sale that month),
    # so it is a real zero - unlike missing values elsewhere, which are never zero-filled.
    for c in ["net", "purchase", "sale", "fwd_book"]:
        is_nil = body[c].astype(str).str.strip().isin(["-", "–"])
        body[c] = pd.to_numeric(body[c].where(~is_nil, 0), errors="coerce")
    body["month"] = pd.to_datetime(body["month"]).dt.to_period("M").dt.to_timestamp()
    out = []
    for col, sid in [("net", "IN_RBI_FX_NET"), ("purchase", "IN_RBI_FX_PURCHASE"),
                     ("sale", "IN_RBI_FX_SALE"), ("fwd_book", "IN_RBI_FWD_BOOK")]:
        out.append(tidy(body, sid, "month", col))
    log("IN_RBI_FX_NET", "definition",
        "net purchase(+)/sale(-) of FX by RBI incl. swap and outright-forward legs, by VALUE date (RBI note)",
        "info", action="monthly; attributed to event windows by calendar month")
    return pd.concat(out)

In [10]:
fx_int = load_rbi_intervention()
fx_int[fx_int.series_id == "IN_RBI_FX_NET"].tail(8)

,series_id,obs_date,value
371,IN_RBI_FX_NET,1996-01-01,-402.00
372,IN_RBI_FX_NET,1995-12-01,-56.00
373,IN_RBI_FX_NET,1995-11-01,-111.00
374,IN_RBI_FX_NET,1995-10-01,-785.00
375,IN_RBI_FX_NET,1995-09-01,0.00
376,IN_RBI_FX_NET,1995-08-01,0.00
377,IN_RBI_FX_NET,1995-07-01,338.18
378,IN_RBI_FX_NET,1995-06-01,36.00


## 5 · FRED: oil, US dollar, risk, US rates, US CPI, reserves, and the Fed's USD/INR

One zip of daily series, one CSV of monthly series and the Fed H.10 USD/INR rate. FRED codes are renamed to
readable IDs (for example `DCOILBRENTEU` → `BRENT`).

In [11]:
RENAME = {"DEXINUS": "USDINR_FRED", "DCOILBRENTEU": "BRENT", "DTWEXBGS": "DXY_BROAD",
              "VIXCLS": "VIX", "DGS10": "UST10Y", "DFEDTARU": "FFR_UPPER", "DFF": "FFR_EFFECTIVE",
              "CPIAUCSL": "US_CPI", "INDCPIALLMINMEI": "IN_CPI_OECD", "TRESEGINM052N": "IN_FX_RESERVES"}


# ---------------------------------------------------------------- FRED
def load_fred():
    frames = []
    z = zipfile.ZipFile(RAW_FRED_DAILY)
    for name in z.namelist():
        if not name.endswith(".csv"):
            continue
        df = pd.read_csv(z.open(name), dtype=str)
        for col in df.columns[1:]:
            frames.append(tidy(df, RENAME.get(col, col), "observation_date", col))
    m = pd.read_csv(RAW_FRED_MONTHLY, dtype=str)
    for col in m.columns[1:]:
        frames.append(tidy(m, RENAME.get(col, col), "observation_date", col))
    dx = pd.read_csv(RAW_FRED_USDINR, dtype=str)
    frames.append(tidy(dx, "USDINR_FRED", "observation_date", "DEXINUS"))
    return pd.concat(frames)

In [12]:
fred = load_fred()
fred.groupby("series_id")["obs_date"].agg(["min", "max", "count"])

,min,max,count
series_id,,,
BRENT,2012-01-03,2026-09-22,3730
DXY_BROAD,2006-01-02,2026-09-25,5198
FFR_EFFECTIVE,1954-07-01,2026-09-28,26388
FFR_UPPER,2008-12-16,2026-09-29,6497
IN_CPI_OECD,1957-01-01,2025-03-01,819
IN_FX_RESERVES,1950-12-01,2026-07-01,842
USDINR_FRED,1973-01-02,2026-09-25,13468
UST10Y,1962-01-02,2026-09-28,16170
US_CPI,2010-01-01,2026-08-01,199


## 6 · Gold: LBMA Gold Price AM (US$/oz)

In [13]:
# ---------------------------------------------------------------- LBMA gold
def load_lbma():
    data = json.loads(RAW_LBMA_GOLD.read_text())
    df = pd.DataFrame({"obs_date": [d["d"] for d in data],
                       "value": [d["v"][0] if d.get("v") else None for d in data]})
    return tidy(df, "GOLD_USD", "obs_date", "value")

In [14]:
gold = load_lbma()
gold.tail(3)

,series_id,obs_date,value
14842,GOLD_USD,2026-09-25,4290.75
14843,GOLD_USD,2026-09-28,4144.40
14844,GOLD_USD,2026-09-29,4139.00


## 7 · NIFTY 50 (yearly files from niftyindices.com)

Yearly downloads share their boundary day (1 January appears in two files), so they are de-duplicated. A **coverage
check** flags any calendar year with fewer than 200 trading days. That check is how a missing 2022 file was caught.

In [15]:
# ---------------------------------------------------------------- NIFTY 50
def load_nifty():
    files = sorted(RAW_NIFTY_DIR.glob("*.csv"))
    if not files:
        log("NIFTY50", "missing_source", "no files in 01_Data/1_Raw/NSE_Nifty50 - equity metrics will be NULL",
            "error", action="download from niftyindices.com and re-run")
        return pd.DataFrame(columns=["series_id", "obs_date", "value"])
    parts = []
    for f in files:
        df = pd.read_csv(f, dtype=str)
        df.columns = [re.sub(r"\s+", " ", c).strip().lower() for c in df.columns]
        dcol = next(c for c in df.columns if "date" in c)
        ccol = next(c for c in df.columns if c.startswith("close"))
        d = pd.DataFrame({"obs_date": pd.to_datetime(df[dcol].str.strip(), dayfirst=True, format="mixed"),
                          "value": df[ccol].str.replace(",", "", regex=False)})
        parts.append(d)
        log("NIFTY50", "file_loaded", f"{f.name}: {len(d)} rows", "info")
    df = pd.concat(parts)
    # yearly downloads share their boundary day (e.g. 01 Jan appears in two files)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    dups = df[df.duplicated("obs_date", keep=False)]
    conflict = dups.groupby("obs_date")["value"].nunique()
    conflict = conflict[conflict > 1]
    for d in conflict.index:
        log("NIFTY50", "conflicting_duplicate", "same date in two files with different closes", "warning",
            d.date(), "kept first file's value")
    n_same = int(df.duplicated("obs_date").sum()) - len(conflict)
    if n_same:
        log("NIFTY50", "boundary_duplicates", f"{n_same} identical rows repeated across yearly files", "info",
            action="de-duplicated")
    df = df.drop_duplicates("obs_date", keep="first")
    # coverage check: flag any calendar year with far fewer rows than ~245 trading days
    per_year = df.groupby(df["obs_date"].dt.year).size()
    last_year = per_year.index.max()
    for y in range(max(per_year.index.min(), 2013), last_year + 1):
        n = int(per_year.get(y, 0))
        if n < 200 and y != last_year:
            log("NIFTY50", "year_coverage_gap", f"{y}: only {n} trading days loaded (expected ~245)",
                "error", action=f"download NIFTY 50 history for {y} into 01_Data/1_Raw/NSE_Nifty50/")
    return tidy(df, "NIFTY50", "obs_date", "value")

In [16]:
nifty = load_nifty()
nifty.groupby(nifty.obs_date.dt.year).size().rename("trading_days").to_frame().T

obs_date,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
trading_days,250,244,248,247,248,246,245,252,248,248,246,249,249,184


## 8 · RBI repo rate (hand-compiled step series from RBI policy statements)

In [17]:
# ---------------------------------------------------------------- RBI repo (step)
def load_repo():
    r = read_reference(REF_REPO)
    return tidy(r, "IN_REPO", "effective_date", "repo_rate_pct")

In [18]:
repo = load_repo()
repo.tail()

,series_id,obs_date,value
27,IN_REPO,2023-02-08,6.50
28,IN_REPO,2025-02-07,6.25
29,IN_REPO,2025-04-09,6.00
30,IN_REPO,2025-06-06,5.50
31,IN_REPO,2025-12-05,5.25


## 9 · Combine, trim to the analysis period, sanity-check

All sources are stacked into one long table. It keeps one extra year before 2013 for returns and rolling statistics,
and drops non-positive prices (none exist, but the check is there). It then cross-checks the two RBI tables.

In [19]:
def crosscheck_rbi_tables(obs):
    """Compare the two RBI tables that both carry the reference rate."""
    a = obs[obs.series_id == "USDINR_RBI"].set_index("obs_date")["value"].sort_index()
    b = obs[obs.series_id == "USDINR_RBI_IBTABLE"].set_index("obs_date")["value"].sort_index()
    both = pd.concat([a.rename("ref"), b.rename("ib")], axis=1, sort=True).dropna()
    d = (both["ref"] - both["ib"]).abs()
    log("USDINR_RBI", "table_crosscheck",
        f"{len(both)} shared dates: {int((d < 1e-9).sum())} identical, {int(((d > 0) & (d <= 0.005)).sum())} "
        f"differ by <= 0.005 (rounding), {int((d > 0.005).sum())} differ by more", "info",
        action="Daily Exchange Rate table used as primary")
    for dt_, row in both[d > 0.01].iterrows():
        log("USDINR_RBI", "table_conflict",
            f"Daily Exchange Rate table {row.ref:.4f} vs interbank table {row.ib:.4f}", "warning",
            dt_.date(), "kept primary value as published; not near any event anchor - review")
    only_ib = b.index.difference(a.index)
    only_ib = only_ib[only_ib >= pd.Timestamp("2013-01-01")]
    copies = [x for x in only_ib if b.index.get_loc(x) > 0 and b.iloc[b.index.get_loc(x) - 1] == b[x]]
    if copies:
        log("USDINR_RBI_IBTABLE", "carried_forward_copies",
            f"{len(copies)} dates (mostly 1 April - banks' annual closing) repeat the previous day's value exactly: "
            + ", ".join(str(x.date()) for x in copies), "info",
            action="excluded - not present in the primary table")

In [20]:
PROC.mkdir(parents=True, exist_ok=True)
obs = pd.concat([rbi_ref, rbi_ib, fred, gold, nifty, repo, cpi, fx_int], ignore_index=True)
obs["obs_date"] = pd.to_datetime(obs["obs_date"])
obs["value"] = pd.to_numeric(obs["value"])

lo = pd.Timestamp(ANALYSIS_START) - pd.DateOffset(years=1)
step = obs["series_id"].isin(["IN_REPO", "FFR_UPPER"])          # policy rates: keep full history
obs = obs[(step | (obs["obs_date"] >= lo)) & (obs["obs_date"] <= pd.Timestamp(ANALYSIS_END))]

price_like = ~obs["series_id"].isin(["UST10Y", "FFR_UPPER", "FFR_EFFECTIVE", "IN_REPO", "IN_RBI_FX_NET",
                                     "IN_RBI_FWD_BOOK", "IN_RBI_FX_PURCHASE", "IN_RBI_FX_SALE"])
bad = obs[price_like & (obs["value"] <= 0)]
for _, r in bad.iterrows():
    log(r.series_id, "non_positive", f"value {r.value}", "error", r.obs_date.date(), "dropped")
obs = obs.drop(bad.index)

crosscheck_rbi_tables(obs)
obs = obs.sort_values(["series_id", "obs_date"])
obs.to_csv(PROC / "observations_long.csv", index=False, date_format="%Y-%m-%d")
pd.DataFrame(issues).to_csv(PROC / "dq_issues_ingest.csv", index=False)

summary = obs.groupby("series_id")["obs_date"].agg(first="min", last="max", observations="count")
summary

,first,last,observations
series_id,,,
BRENT,2012-01-03,2026-09-22,3730
DXY_BROAD,2012-01-03,2026-09-25,3670
FFR_EFFECTIVE,2012-01-01,2026-09-28,5385
FFR_UPPER,2008-12-16,2026-09-29,6497
GOLD_USD,2012-01-03,2026-09-29,3725
IN_CPI_MOSPI,2012-01-01,2026-08-01,176
IN_CPI_OECD,2012-01-01,2025-03-01,159
IN_FX_RESERVES,2012-01-01,2026-07-01,175
IN_RBI_FWD_BOOK,2012-01-01,2026-07-01,175


## 10 · The issue log

Every decision made while reading the data. This table feeds the **Data Quality** page of the dashboard.

In [21]:
issue_log = pd.DataFrame(issues)
print(issue_log.severity.value_counts().to_string())
issue_log[issue_log.severity != "info"][["series_id", "obs_date", "rule", "detail", "action"]]

severity
info       36
warning     7


,series_id,obs_date,rule,detail,action
2,USDINR_RBI_IBTABLE,2025-12-20,weekend_value,value on a Saturday in the interbank table,not used - primary series comes from the Daily Exchange Rate table
12,IN_CPI_MOSPI,None,series_break,CPI base changes 2010->2012 (Jan-2014) and 2012->2024 (Jan-2026); index LEVE...,base year stored per row (fact_macro.series_version)
37,USDINR_RBI,2013-07-18,table_conflict,Daily Exchange Rate table 60.0510 vs interbank table 59.7100,kept primary value as published; not near any event anchor - review
38,USDINR_RBI,2019-10-29,table_conflict,Daily Exchange Rate table 70.7303 vs interbank table 70.9523,kept primary value as published; not near any event anchor - review
39,USDINR_RBI,2019-10-30,table_conflict,Daily Exchange Rate table 70.9915 vs interbank table 70.7303,kept primary value as published; not near any event anchor - review
40,USDINR_RBI,2019-10-31,table_conflict,Daily Exchange Rate table 70.8095 vs interbank table 70.9915,kept primary value as published; not near any event anchor - review
41,USDINR_RBI,2019-11-01,table_conflict,Daily Exchange Rate table 70.8798 vs interbank table 70.8095,kept primary value as published; not near any event anchor - review


In [22]:
# the two RBI tables compared
issue_log[issue_log.rule.isin(["table_crosscheck", "carried_forward_copies"])][["rule", "detail", "action"]]

,rule,detail,action
36,table_crosscheck,"3472 shared dates: 2815 identical, 628 differ by <= 0.005 (rounding), 29 dif...",Daily Exchange Rate table used as primary
42,carried_forward_copies,11 dates (mostly 1 April - banks' annual closing) repeat the previous day's ...,excluded - not present in the primary table


**Output:** `01_Data/3_Intermediate/observations_long.csv` and `dq_issues_ingest.csv`. Next: **02 · Align & transform**.